In [36]:
import re
import pandas as pd
import numpy as np

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv(r"C:\Users\devan\Desktop\Coding Folders\Python\Positiveway Internship\spam.csv", encoding="latin-1")

# Keep only needed columns
df = df[['v1','v2']].rename(columns={'v1':'label','v2':'text'})

# Map labels to integers
df['label'] = df['label'].map({'ham':0, 'spam':1})
df = df.dropna()

texts = df['text'].astype(str).values
labels = df['label'].astype(np.int32).values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(X_train, y_train)

['Going on nothing great.bye' "I wont. So wat's wit the guys"
 'Ok k..sry i knw 2 siva..tats y i askd..' ...
 'Moby Pub Quiz.Win a å£100 High Street prize if u know who the new Duchess of Cornwall will be? Txt her first name to 82277.unsub STOP å£1.50 008704050406 SP Arrow'
 "Free entry in 2 a weekly comp for a chance to win an ipod. Txt POD to 80182 to get entry (std txt rate) T&C's apply 08452810073 for details 18+"
 'Nothing but we jus tot u would ask cos u ba gua... But we went mt faber yest... Yest jus went out already mah so today not going out... Jus call lor...'] [0 0 0 ... 1 1 0]


In [37]:
def clean_text(s: tf.Tensor) -> tf.Tensor:
    s = tf.strings.lower(s)
    # remove weird punctuation but keep letters/numbers/spaces
    s = tf.strings.regex_replace(s, r"[^a-z0-9\s]", " ")
    s = tf.strings.regex_replace(s, r"\s+", " ")
    return tf.strings.strip(s)

# Wrap it for tf.data
def tf_clean_text(x, y):
    return clean_text(x), y


In [38]:
max_tokens = 20000
sequence_length = 200  # messages are short; adjust if needed

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Fit vectorizer on training text
vectorizer.adapt(X_train)

In [39]:
vocab_size = max_tokens

model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=64),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [40]:
batch_size = 32
epochs = 10

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(batch_size).map(tf_clean_text).prefetch(tf.data.AUTOTUNE)
test_ds  = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).map(tf_clean_text).prefetch(tf.data.AUTOTUNE)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=epochs
)

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8638 - loss: 0.3966 - val_accuracy: 0.8664 - val_loss: 0.3752
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8658 - loss: 0.3758 - val_accuracy: 0.8664 - val_loss: 0.3599
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8658 - loss: 0.3518 - val_accuracy: 0.8664 - val_loss: 0.3172
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8750 - loss: 0.2879 - val_accuracy: 0.9184 - val_loss: 0.4008
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9367 - loss: 0.1807 - val_accuracy: 0.9094 - val_loss: 0.3088
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9708 - loss: 0.1018 - val_accuracy: 0.9238 - val_loss: 0.2387
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9755 - loss: 0.0930 - val_accuracy: 0.8969 - val_loss: 0.3003
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9798 - loss: 0.0743 - val_accuracy: 0

In [41]:
# Predictions
y_pred_prob = model.predict(test_ds).ravel()
y_pred = (y_pred_prob >= 0.6).astype(int)

print(classification_report(y_test, y_pred, target_names=["ham","spam"]))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       966
        spam       0.86      0.93      0.90       149

    accuracy                           0.97      1115
   macro avg       0.93      0.96      0.94      1115
weighted avg       0.97      0.97      0.97      1115

Confusion matrix:
[[944  22]
 [ 10 139]]


In [42]:
def predict_message(msg: str):
    # model expects raw string since vectorizer is inside model
    # convert input to a tf.Tensor of strings so the TextVectorization layer accepts it
    inp = tf.constant([msg])
    prob = float(model.predict(inp, verbose=0).ravel()[0])
    label = "spam" if prob >= 0.5 else "ham"
    return label, prob

label, prob = predict_message("You account has been given a free gift card")
print(label, f",The probability that it may be a spam msg is: {prob}")

ham ,The probability that it may be a spam msg is: 0.15749701857566833


In [50]:
# Save the trained model (SavedModel format)
model_dir = r"C:\Users\devan\Desktop\Coding Folders\Python\Positiveway Internship\spam_model"
model.export("spam_model")
print('Saved model to', model_dir)

INFO:tensorflow:Assets written to: spam_model\assets


INFO:tensorflow:Assets written to: spam_model\assets


Saved artifact at 'spam_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None,), dtype=tf.string, name='keras_tensor_7')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2513920905232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2513920906576: TensorSpec(shape=(), dtype=tf.int64, name=None)
  2513920532432: TensorSpec(shape=(), dtype=tf.string, name=None)
  2513920528784: TensorSpec(shape=(), dtype=tf.int64, name=None)
  2513920903696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2513920914256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2513920911760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2513920906000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2513920909840: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved model to C:\Users\devan\Desktop\Coding Folders\Python\Positiveway Internship\spam_model


In [51]:
loaded = tf.keras.layers.TFSMLayer(
    "spam_model",
    call_endpoint="serve"
)

def predict_with_loaded(msgs):
    inp = tf.constant(msgs)
    probs = loaded(inp).numpy().ravel()
    labels = ['spam' if p >= 0.5 else 'ham' for p in probs]
    return list(zip(msgs, probs, labels))

print(predict_with_loaded([
    'Free entry in 2 a wkly comp...',
    'Hey, are we meeting today?'
]))

[('Free entry in 2 a wkly comp...', np.float32(0.16576542), 'ham'), ('Hey, are we meeting today?', np.float32(0.006261951), 'ham')]


In [43]:
print(type(model))
print(model.input_shape)
model.summary()

<class 'keras.src.models.sequential.Sequential'>
(None,)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 200)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 200, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,852,677 (14.70 MB)

 Trainable params: 1,284,225 (4.90 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,568,452 (9.80 MB)